# G8 — training-free similarity: mutual k-NN, CKA, relative representations

Three ways to ask *"do these two encoders agree?"* without fitting a map, run over the
same cached spaces the hub uses. Each one is paired with the control that makes it
falsifiable.

| statistic | what it is invariant to | chance floor |
|---|---|---|
| **mutual k-NN** (Huh et al., PRH) | any transform preserving the cosine neighbour order | `k / (n-1)` — **not zero** |
| **linear CKA**, biased | orthogonal transforms, isotropic scaling | `~ d / n` — **not zero** |
| **linear CKA**, debiased (unbiased HSIC) | same | `~ 0`, may go slightly negative |
| **relative representations** (Moschella et al.) | per-encoder rotation; widths need not match | measured by shuffle |
| **shape agreement** (this project's ρ) | monotone rescalings of cosine | `~ 0` |

Two project constraints are wired in rather than assumed:

1. **Row alignment is positional, never by id.** Cell 5 verifies it the way `G4_convnext`
   and `H1` do — the statistic must vastly exceed its own row-shuffled value — and refuses
   to continue on a pair that fails.
2. **Stratify before averaging.** Cell 6 flags degenerate spaces (the GPT-2 caption space)
   by *algebraic* rank and anisotropy, kept separate from entropy-based effective rank.
   Every mean below is reported both pooled and stratified.

Nothing here refits the hub or touches `adapter.npz`.

## 1 — Storage cell (portable: Colab Drive, or a local path)

In [ ]:
import os, sys, glob
from pathlib import Path

try:
    from google.colab import drive          # noqa
    drive.mount("/content/drive", force_remount=False)
    DATA_DIR = Path("/content/drive/MyDrive/convergence_experiment")
except Exception:
    DATA_DIR = Path(os.environ.get("DATA_DIR", "."))

OUT_DIR = Path(os.environ.get("OUT_DIR", DATA_DIR / "G8_outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| exists:", DATA_DIR.exists())
print("OUT_DIR :", OUT_DIR)
for p in sorted(glob.glob(str(DATA_DIR / "**" / "*.npz"), recursive=True))[:40]:
    print("   ", Path(p).relative_to(DATA_DIR), f"{os.path.getsize(p)/1e6:.1f} MB")

## 2 — Metrics (self-contained; no dependency beyond numpy/scipy)

In [ ]:
"""Representation-comparison metrics: mutual k-NN, CKA, relative representations,
plus the project's Spearman shape-agreement statistic and shuffle controls.

Conventions
-----------
* Every function takes X: (n, d1) and Y: (n, d2) with ROW i in X and ROW i in Y
  describing the SAME item. Row correspondence is positional (see README note on
  cache alignment); it is never inferred from ids.
* Nothing here L2-normalizes or centers behind your back except where the metric
  definition requires it. The geometry you pass in is the geometry measured.
"""
import numpy as np
from scipy.stats import spearmanr


# ---------------------------------------------------------------- preprocessing
def l2n(X, eps=1e-12):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)


def center(X):
    return X - X.mean(0, keepdims=True)


def pca_whiten(X, k=None, eps=1e-6):
    """Zero-mean, unit-variance in the PCA basis. k=None keeps min(n-1, d)."""
    Xc = center(X)
    U, S, _ = np.linalg.svd(Xc, full_matrices=False)
    n = Xc.shape[0]
    if k is None:
        k = min(n - 1, Xc.shape[1])
    k = min(k, S.shape[0])
    return U[:, :k] * np.sqrt(n - 1) * (S[:k] > eps * S[0])


def effective_rank(X, eps=1e-12):
    """Entropy-based effective rank. NOT the algebraic rank -- the project's
    G13 error was conflating the two. Reported alongside, never instead."""
    s = np.linalg.svd(center(X), compute_uv=False)
    p = s / (s.sum() + eps)
    p = p[p > eps]
    return float(np.exp(-(p * np.log(p)).sum()))


def algebraic_rank(X, tol=None):
    return int(np.linalg.matrix_rank(center(X), tol=tol))


# ---------------------------------------------------------------- mutual k-NN
def knn_indices(X, k, metric="cosine"):
    """Indices of the k nearest neighbours of each row, self excluded."""
    if metric == "cosine":
        S = l2n(X) @ l2n(X).T
    elif metric == "euclidean":
        sq = (X ** 2).sum(1)
        S = -(sq[:, None] + sq[None, :] - 2 * X @ X.T)
    else:
        raise ValueError(metric)
    np.fill_diagonal(S, -np.inf)
    # argpartition then sort the top-k block: O(n) per row instead of O(n log n)
    part = np.argpartition(-S, kth=k, axis=1)[:, :k]
    rows = np.arange(X.shape[0])[:, None]
    order = np.argsort(-S[rows, part], axis=1)
    return part[rows, order]


def mutual_knn(X, Y, k=10, metric="cosine"):
    """Huh et al. (PRH) alignment: mean overlap of k-NN sets, in [0, 1].

    Chance level is k / (n - 1), which is NOT zero -- always report it.
    """
    nx, ny = knn_indices(X, k, metric), knn_indices(Y, k, metric)
    n = X.shape[0]
    masks = np.zeros((n, n), dtype=bool)
    masks[np.arange(n)[:, None], nx] = True
    overlap = masks[np.arange(n)[:, None], ny].sum(1)
    return float(overlap.mean() / k)


def mutual_knn_chance(n, k):
    return k / (n - 1)


# ---------------------------------------------------------------- CKA
def cka_linear(X, Y):
    """Linear CKA (Kornblith et al.). Invariant to orthogonal transforms and
    isotropic scaling; NOT invariant to per-feature rescaling, which is why the
    raw and whitened numbers below are different measurements, not duplicates."""
    Xc, Yc = center(X), center(Y)
    num = np.linalg.norm(Xc.T @ Yc, ord="fro") ** 2
    den = np.linalg.norm(Xc.T @ Xc, ord="fro") * np.linalg.norm(Yc.T @ Yc, ord="fro")
    return float(num / (den + 1e-12))


def _hsic1(K, L):
    """Unbiased HSIC (Song et al. 2012). Needed because the standard biased
    estimator gives CKA a floor of roughly d/n for INDEPENDENT spaces -- with
    d=768 and n=9533 that floor is ~0.08, which is not negligible when you are
    comparing weak cross-modal pairs."""
    n = K.shape[0]
    K = K.copy(); L = L.copy()
    np.fill_diagonal(K, 0.0); np.fill_diagonal(L, 0.0)
    ones = np.ones(n)
    t1 = np.trace(K @ L)
    t2 = (ones @ K @ ones) * (ones @ L @ ones) / ((n - 1) * (n - 2))
    t3 = 2.0 / (n - 2) * (ones @ K @ L @ ones)
    return float((t1 + t2 - t3) / (n * (n - 3)))


def _hsic1_linear(Xc, Yc):
    """Exact closed form of _hsic1 for linear kernels on CENTERED features.
    O(n*d^2) instead of O(n^3): never materializes an n x n Gram matrix, which
    matters at n = 9,533 (a float64 Gram is 727 MB, and two of them plus a
    matmul will not fit comfortably on Colab)."""
    n = Xc.shape[0]
    a = (Xc ** 2).sum(1)          # K_ii
    b = (Yc ** 2).sum(1)          # L_ii
    ab = float((a * b).sum())
    t1 = float(np.linalg.norm(Xc.T @ Yc, ord="fro") ** 2) - ab
    t2 = float(a.sum()) * float(b.sum()) / ((n - 1) * (n - 2))
    t3 = 2.0 / (n - 2) * ab
    return float((t1 + t2 - t3) / (n * (n - 3)))


def cka_linear_debiased(X, Y):
    """Linear CKA built on the unbiased HSIC estimator. Floor is ~0 for
    independent spaces; can go slightly negative, which is correct, not a bug."""
    Xc, Yc = center(X), center(Y)
    num = _hsic1_linear(Xc, Yc)
    den = np.sqrt(max(_hsic1_linear(Xc, Xc), 0.0) * max(_hsic1_linear(Yc, Yc), 0.0))
    return float(num / (den + 1e-12))


def _rbf_gram(X, sigma_frac=0.8):
    sq = (X ** 2).sum(1)
    D2 = np.maximum(sq[:, None] + sq[None, :] - 2 * X @ X.T, 0.0)
    med = np.median(D2[D2 > 0]) if (D2 > 0).any() else 1.0
    return np.exp(-D2 / (2 * (sigma_frac ** 2) * med + 1e-12))


def _center_gram(K):
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    return H @ K @ H


def cka_rbf(X, Y, sigma_frac=0.8):
    """Kernel CKA with an RBF kernel at a median-heuristic bandwidth."""
    Kx, Ky = _center_gram(_rbf_gram(X, sigma_frac)), _center_gram(_rbf_gram(Y, sigma_frac))
    num = (Kx * Ky).sum()
    den = np.sqrt((Kx * Kx).sum() * (Ky * Ky).sum())
    return float(num / (den + 1e-12))


# ------------------------------------------------- relative representations
def relative_representation(X, anchor_idx, metric="cosine"):
    """Moschella et al.: re-express each row by its similarity to a fixed anchor
    set. Output is (n, n_anchors) and lives in the SAME space for every encoder,
    so encoders of different widths become directly comparable."""
    if metric == "cosine":
        return l2n(X) @ l2n(X[anchor_idx]).T
    if metric == "euclidean":
        d = np.linalg.norm(X[:, None, :] - X[None, anchor_idx, :], axis=2)
        return -d
    raise ValueError(metric)


def relative_agreement(X, Y, anchor_idx, metric="cosine"):
    """Mean per-item cosine between the two relative representations.

    Anchors are shared rows, so this needs no fitted map -- it is the
    training-free counterpart to the hub. Chance is ~0 for centered anchors but
    is measured, not assumed, by the shuffle control."""
    Rx = l2n(center(relative_representation(X, anchor_idx, metric)))
    Ry = l2n(center(relative_representation(Y, anchor_idx, metric)))
    return float((Rx * Ry).sum(1).mean())


# ------------------------------------------------- shape agreement (project's rho)
def shape_agreement(X, Y, sample=None, rng=None):
    """Spearman rho between the two off-diagonal cosine-similarity matrices --
    the statistic behind the project's A15b matrix (bge-SBERT 0.768)."""
    rng = np.random.default_rng(0) if rng is None else rng
    n = X.shape[0]
    idx = np.arange(n) if (sample is None or sample >= n) else rng.choice(n, sample, replace=False)
    Sx = l2n(X[idx]) @ l2n(X[idx]).T
    Sy = l2n(Y[idx]) @ l2n(Y[idx]).T
    iu = np.triu_indices(len(idx), k=1)
    return float(spearmanr(Sx[iu], Sy[iu]).statistic)


# ---------------------------------------------------------------- controls
def shuffle_control(fn, X, Y, n_rep=5, seed=0, **kw):
    """Break row correspondence and re-measure. Any statistic that does not
    collapse here is not measuring shared structure."""
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_rep):
        perm = rng.permutation(X.shape[0])
        vals.append(fn(X, Y[perm], **kw))
    return float(np.mean(vals)), float(np.std(vals))

## 3 — Self-test on synthetic data — **run this before touching the caches**

Known-answer checks: identity → 1, rotation → 1, independent → the stated floor,
monotone decay with noise, shuffle collapses, whitening gives identity covariance,
algebraic rank ≠ effective rank. If any line says FAIL, stop — the numbers below
would be meaningless.

In [ ]:
from scipy.stats import ortho_group
import numpy as np
from scipy.stats import ortho_group

rng = np.random.default_rng(0)
n, d = 400, 64
X = rng.normal(size=(n, d))
Q = ortho_group.rvs(d, random_state=1)
Z = rng.normal(size=(n, d))          # independent
anch = rng.choice(n, 32, replace=False)
fails = []


def chk(name, got, want, tol):
    ok = abs(got - want) <= tol
    print(f"{'PASS' if ok else 'FAIL'}  {name:<44} got={got:.4f} want~{want:.4f}")
    if not ok:
        fails.append(name)


# --- identity ---------------------------------------------------------------
chk("mutual_knn(X,X)", mutual_knn(X, X, 10), 1.0, 1e-9)
chk("cka_linear(X,X)", cka_linear(X, X), 1.0, 1e-9)
chk("cka_rbf(X,X)", cka_rbf(X, X), 1.0, 1e-9)
chk("relative_agreement(X,X)", relative_agreement(X, X, anch), 1.0, 1e-9)
chk("shape_agreement(X,X)", shape_agreement(X, X), 1.0, 1e-9)

# --- invariances that MUST hold --------------------------------------------
chk("mutual_knn rotation-invariant", mutual_knn(X, X @ Q, 10), 1.0, 1e-9)
chk("cka_linear rotation-invariant", cka_linear(X, X @ Q), 1.0, 1e-9)
chk("cka_linear isotropic-scale-invariant", cka_linear(X, 7.3 * X), 1.0, 1e-9)
chk("relative_agreement rotation-invariant", relative_agreement(X, X @ Q, anch), 1.0, 1e-9)
chk("shape_agreement rotation-invariant", shape_agreement(X, X @ Q), 1.0, 1e-9)

# --- chance levels ----------------------------------------------------------
chk("mutual_knn independent ~ k/(n-1)", mutual_knn(X, Z, 10), mutual_knn_chance(n, 10), 0.02)
chk("cka_linear independent ~ d/n (BIASED)", cka_linear(X, Z), d / n, 0.05)
chk("cka_linear_debiased independent ~ 0", cka_linear_debiased(X, Z), 0.0, 0.02)
chk("cka_linear_debiased(X,X)", cka_linear_debiased(X, X), 1.0, 1e-6)
chk("cka_linear_debiased rotation-inv", cka_linear_debiased(X, X @ Q), 1.0, 1e-6)
chk("relative_agreement independent ~ 0", relative_agreement(X, Z, anch), 0.0, 0.10)
chk("shape_agreement independent ~ 0", shape_agreement(X, Z), 0.0, 0.10)

# --- monotonicity in shared signal -----------------------------------------
S = rng.normal(size=(n, d))
prev = {}
for noise in (0.1, 0.5, 1.0, 2.0):
    A = S + noise * rng.normal(size=(n, d))
    B = S @ Q + noise * rng.normal(size=(n, d))
    vals = dict(knn=mutual_knn(A, B, 10), cka=cka_linear_debiased(A, B),
                rel=relative_agreement(A, B, anch), rho=shape_agreement(A, B, sample=250))
    print(f"      noise={noise:<4} " + "  ".join(f"{k}={v:.3f}" for k, v in vals.items()))
    for k, v in vals.items():
        if prev and v > prev[k] + 1e-9:
            fails.append(f"monotonicity/{k}@{noise}")
            print(f"FAIL  monotonicity broken for {k}")
    prev = vals
print("PASS  all four metrics decrease monotonically with noise"
      if not any(f.startswith("monotonicity") for f in fails) else "FAIL  monotonicity")

# --- shuffle control collapses ---------------------------------------------
A = S + 0.3 * rng.normal(size=(n, d))
B = S @ Q + 0.3 * rng.normal(size=(n, d))
m, s = shuffle_control(mutual_knn, A, B, n_rep=3, k=10)
chk("shuffled mutual_knn ~ chance", m, mutual_knn_chance(n, 10), 0.02)
m2, _ = shuffle_control(cka_linear, A, B, n_rep=3)
chk("shuffled cka_linear ~ d/n (BIASED)", m2, d / n, 0.05)
m3, _ = shuffle_control(cka_linear_debiased, A, B, n_rep=3)
chk("shuffled cka_linear_debiased ~ 0", m3, 0.0, 0.02)
print(f"      intact: knn={mutual_knn(A,B,10):.3f} cka={cka_linear(A,B):.3f}")

# --- whitening behaves ------------------------------------------------------
W = pca_whiten(X)
cov = (W.T @ W) / (n - 1)
chk("pca_whiten -> identity covariance", float(np.abs(cov - np.eye(cov.shape[0])).max()), 0.0, 1e-8)

# --- rank diagnostics distinguish algebraic from entropy rank ---------------
Xlow = rng.normal(size=(n, 5)) @ rng.normal(size=(5, d))
aniso = X * np.r_[100.0, np.full(d - 1, 0.01)]
print(f"      low-rank(5): alg={algebraic_rank(Xlow)} eff={effective_rank(Xlow):.2f}")
print(f"      anisotropic: alg={algebraic_rank(aniso)} eff={effective_rank(aniso):.2f}")
if algebraic_rank(Xlow) != 5:
    fails.append("algebraic_rank")
if not (effective_rank(aniso) < 2.0 and algebraic_rank(aniso) == d):
    fails.append("eff_vs_alg_rank")
print("PASS  algebraic rank != effective rank on an anisotropic full-rank space"
      if "eff_vs_alg_rank" not in fails else "FAIL  rank diagnostics")

print("\n" + ("ALL TESTS PASSED" if not fails else f"FAILURES: {fails}"))

## 4 — Register the encoders

Edit `SPECS` only. Each entry is `name -> (file, key)`; the cell prints every array in
every `.npz` it can open first, so you can read the right keys off the listing rather
than guessing. `modality` and `family` drive the stratified means later.

In [ ]:
import numpy as np

# --- what is actually in the caches -----------------------------------------
for p in sorted(DATA_DIR.rglob("*.npz")):
    try:
        with np.load(p, allow_pickle=False) as z:
            print(p.name)
            for k in z.files:
                print(f"    {k:<24} {z[k].shape} {z[k].dtype}")
    except Exception as e:
        print(p.name, "-> unreadable:", type(e).__name__, e)

# --- EDIT ME ------------------------------------------------------------------
# The canonical 8-encoder roster (report E.10/E.12), file GLOBS + keys matching G9/G10.
# SigLIP was HELD OUT of this roster; BERT is in it. Do not re-add siglip here.
# name: (filename glob, preferred key, modality, family)
SPECS = {
    "img_small": ("e1_img_ckpt_dinov2-small_cls+patch*", "img", "image", "dinov2"),
    "img_base":  ("e1_img_ckpt_dinov2-base_cls+patch*",  "img", "image", "dinov2"),
    "img_large": ("e1_img_ckpt_dinov2-large_cls+patch*", "img", "image", "dinov2"),
    "convnext":  ("e1_img_ckpt_convnext-base-224-22k*",  "img", "image", "convnext"),
    "bge":       ("crossmodal_pairs.npz",                "txt", "text",  "contrastive-text"),
    "gpt2":      ("crossmodal_pairs_gpt2.npz",           "txt", "text",  "causal-lm"),
    "bert":      ("e13_txt_bert*",                       "txt", "text",  "masked-lm"),
    "sbert":     ("e13_txt_sbert*",                      "txt", "text",  "contrastive-text"),
}
N_ITEMS = 9533     # the hub corpus; every cache is truncated to this prefix (E.15)

In [ ]:
import fnmatch

def _find(glob_pat):
    hits = sorted(p for p in DATA_DIR.rglob("*") if p.is_file()
                  and fnmatch.fnmatch(p.name, glob_pat) and p.suffix == ".npz")
    return hits

def _pick_key(z, preferred):
    if preferred in z.files and z[preferred].ndim == 2:
        return preferred
    cands = [k for k in z.files if z[k].ndim == 2 and np.issubdtype(z[k].dtype, np.floating)]
    if len(cands) == 1:
        return cands[0]
    raise KeyError(f"ambiguous key; 2-D float arrays = {cands}")

E, META, missing = {}, {}, []
for name, (glob_pat, key, modality, family) in SPECS.items():
    hits = _find(glob_pat)
    if not hits:
        missing.append(f"{name}: no file matching '{glob_pat}'"); continue
    if len(hits) > 1:
        print(f"  note: {name} matched {len(hits)} files, using {hits[0].name}: {[h.name for h in hits]}")
    with np.load(hits[0], allow_pickle=False) as z:
        try:
            k = _pick_key(z, key)
        except KeyError as e:
            missing.append(f"{name}: {e} in {hits[0].name}"); continue
        X = np.asarray(z[k], dtype=np.float64)
    if X.shape[0] < N_ITEMS:
        missing.append(f"{name}: {X.shape[0]} rows < N_ITEMS={N_ITEMS}"); continue
    E[name] = X[:N_ITEMS]
    META[name] = dict(modality=modality, family=family, file=hits[0].name, key=k)

if missing:
    print("!! UNRESOLVED — fix SPECS before continuing:")
    for m in missing: print("   ", m)

n_rows = {k: v.shape[0] for k, v in E.items()}
if len(set(n_rows.values())) > 1:
    print("\n!! ROW COUNTS DIFFER after truncation — should not happen at N_ITEMS:", n_rows)

print(f"\nloaded {len(E)} of 8 encoders (all truncated to {N_ITEMS} rows)")
for k, v in E.items():
    print(f"  {k:<10} {str(v.shape):<14} {META[k]['modality']:<6} {META[k]['family']:<18} {META[k]['file']}")
NAMES = list(E)
N = next(iter(E.values())).shape[0]

## 5 — Alignment gate

`keep` values are meaningless across cache generations (E1.1-era caches store the request
position; G4-era caches store real COCO ids). The only valid check is statistical: the
intact Spearman must vastly exceed the row-shuffled one. A pair whose ratio falls below
`MIN_RATIO` is recorded and **excluded** from every matrix below.

In [ ]:
import itertools, pandas as pd

ALIGN_SAMPLE = 1500     # rows used for the gate; the gate is cheap, not the science
MIN_RATIO    = 5.0      # intact |rho| must exceed shuffled |rho| by this factor
rng = np.random.default_rng(0)

rows, blocked = [], set()
for a, b in itertools.combinations(NAMES, 2):
    r  = shape_agreement(E[a], E[b], sample=ALIGN_SAMPLE, rng=np.random.default_rng(0))
    rs, _ = shuffle_control(lambda X, Y: shape_agreement(X, Y, sample=ALIGN_SAMPLE,
                                                         rng=np.random.default_rng(0)),
                            E[a], E[b], n_rep=3, seed=1)
    ratio = abs(r) / max(abs(rs), 1e-4)
    ok = ratio >= MIN_RATIO
    if not ok:
        blocked.add((a, b))
    rows.append(dict(pair=f"{a}~{b}", rho=r, rho_shuffled=rs, ratio=ratio, aligned=ok))

gate = pd.DataFrame(rows).sort_values("ratio")
print(gate.to_string(index=False, float_format=lambda x: f"{x:8.4f}"))
print(f"\n{len(blocked)} pair(s) failed the alignment gate"
      + (": " + ", ".join(f"{a}~{b}" for a, b in blocked) if blocked else ""))
print("NOTE: a failure here can mean misalignment OR genuinely unrelated spaces."
      "\n      Cross-modal pairs are expected to be weak; check those by hand before"
      "\n      concluding the rows are shuffled.")

## 6 — Per-encoder geometry: the stratification flag

Algebraic rank and entropy-based effective rank are different quantities — conflating them
was the project's own G13 error, and it is in the falsification ledger. Both are printed.
A space is flagged degenerate when its effective rank is a small fraction of its ambient
width; flagged spaces are averaged separately, never pooled in.

In [ ]:
DEGENERATE_FRAC = 0.05          # eff_rank / ambient below this -> flagged (low-rank criterion)
ANISO_COS       = 0.95          # mean pairwise cosine above this -> flagged (GENUINE collapse only).
                                # Text encoders sit naturally higher than image ones (bert 0.81,
                                # bge 0.57) WITHOUT being degenerate; only true collapse (GPT-2 0.999)
                                # must trip this. A low cutoff (e.g. 0.30) wrongly flags healthy text
                                # encoders and corrupts the 'healthy-only' stratified means.
ALWAYS_FLAG     = {"gpt2"}      # documented collapse (report C.11/E.10) -- flagged regardless

geo = []
for k in NAMES:
    X = E[k]
    s = np.linalg.svd(center(X), compute_uv=False)
    eff, alg = effective_rank(X), algebraic_rank(X)
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    _mpc_idx = np.random.default_rng(0).choice(len(Xn), min(3000, len(Xn)), replace=False)
    _G = Xn[_mpc_idx] @ Xn[_mpc_idx].T
    mpc = float(_G[np.triu_indices(len(_mpc_idx), 1)].mean())
    low_rank = eff / X.shape[1] < DEGENERATE_FRAC
    anisotropic = mpc > ANISO_COS
    geo.append(dict(encoder=k, n=X.shape[0], ambient=X.shape[1], alg_rank=alg,
                    eff_rank=round(eff, 1), eff_frac=round(eff / X.shape[1], 4),
                    mean_pair_cos=round(mpc, 4),
                    top1_var=round(float((s[0]**2) / (s**2).sum()), 4),
                    mean_norm=round(float(np.linalg.norm(X, axis=1).mean()), 3),
                    low_rank=low_rank, anisotropic=anisotropic,
                    degenerate=(low_rank or anisotropic or k in ALWAYS_FLAG)))
geo = pd.DataFrame(geo)
print(geo.to_string(index=False))
FLAGGED = set(geo.loc[geo.degenerate, "encoder"]) | ALWAYS_FLAG
print(f"\nflagged degenerate: {FLAGGED or 'none'}")
print(f"  criteria: eff_rank/ambient < {DEGENERATE_FRAC} (low-rank) OR mean_pair_cos > {ANISO_COS} "
      f"(anisotropy) OR in {ALWAYS_FLAG} (documented). GPT-2's problem is anisotropy, which a "
      f"centered-eff-rank rule alone can miss -- so both criteria are reported and combined.")

## 7 — The four matrices, in two geometries

Raw and PCA-whitened are **different measurements of different objects**, not a robustness
check — C.12 already showed whitening helps you read a known correspondence and hurts you
when you must find an unknown one. Both are computed so that claim stays testable here.

In [ ]:
K_NN        = 10
N_ANCHORS   = 512
SUBSAMPLE   = 4000      # rows for the O(n^2) statistics; None = all rows
WHITEN_DIM  = 256

rng = np.random.default_rng(0)
idx = np.arange(N) if (SUBSAMPLE is None or SUBSAMPLE >= N) else np.sort(rng.choice(N, SUBSAMPLE, replace=False))
anchor_idx = rng.choice(len(idx), N_ANCHORS, replace=False)

VIEWS = {
    "raw":      {k: l2n(E[k][idx]) for k in NAMES},
    # row-normalize the whitened scores as well, so raw and whitened differ ONLY by whitening.
    # Without this, CKA (which is not cosine-based and does not self-normalize) would compare
    # L2-normalized rows in 'raw' against non-normalized whitened scores in 'whitened' -- more
    # than whitening would change between the two conditions (reviewer point #5).
    "whitened": {k: l2n(pca_whiten(E[k][idx], k=WHITEN_DIM)) for k in NAMES},
}
print(f"n={len(idx)} rows, k={K_NN}, anchors={N_ANCHORS}, whitened to {WHITEN_DIM}d")
print(f"mutual-kNN chance floor = {mutual_knn_chance(len(idx), K_NN):.4f}")

STATS = {
    "mutual_knn": lambda X, Y: mutual_knn(X, Y, K_NN),
    "cka_debiased": cka_linear_debiased,
    "cka_biased": cka_linear,
    "relative": lambda X, Y: relative_agreement(X, Y, anchor_idx),
    "shape_rho": lambda X, Y: shape_agreement(X, Y, sample=2000, rng=np.random.default_rng(0)),
}

M = {}
for view, spaces in VIEWS.items():
    for sname, fn in STATS.items():
        A = pd.DataFrame(np.eye(len(NAMES)), index=NAMES, columns=NAMES, dtype=float)
        for a, b in itertools.combinations(NAMES, 2):
            v = np.nan if (a, b) in blocked else fn(spaces[a], spaces[b])
            A.loc[a, b] = A.loc[b, a] = v
        M[(view, sname)] = A
        print(f"  done {view:<9} {sname}")

In [ ]:
for (view, sname), A in M.items():
    if sname == "cka_biased":
        continue
    print(f"\n=== {sname}  [{view}] ===")
    print(A.to_string(float_format=lambda x: f"{x:6.3f}"))

### Floors, measured rather than assumed

The floor for each statistic is re-measured under a row shuffle on a few representative
pairs. `cka_biased` is included here precisely to show how far its floor sits from zero.

In [ ]:
probe = [p for p in itertools.combinations(NAMES, 2) if p not in blocked][:4]
flo = []
for a, b in probe:
    for sname, fn in STATS.items():
        m, s = shuffle_control(fn, VIEWS["raw"][a], VIEWS["raw"][b], n_rep=3, seed=2)
        flo.append(dict(pair=f"{a}~{b}", stat=sname, intact=fn(VIEWS["raw"][a], VIEWS["raw"][b]),
                        shuffled=m, shuffled_sd=s))
flo = pd.DataFrame(flo)
print(flo.pivot(index="pair", columns="stat", values=["intact", "shuffled"])
        .to_string(float_format=lambda x: f"{x:7.4f}"))

## 8 — Figures

In [ ]:
import matplotlib.pyplot as plt

show = [("raw", "mutual_knn"), ("raw", "cka_debiased"), ("raw", "relative"), ("raw", "shape_rho"),
        ("whitened", "mutual_knn"), ("whitened", "cka_debiased"), ("whitened", "relative"),
        ("whitened", "shape_rho")]
fig, axes = plt.subplots(2, 4, figsize=(21, 10))
for ax, key in zip(axes.ravel(), show):
    A = M[key].values.astype(float)
    im = ax.imshow(A, vmin=np.nanmin(A), vmax=1.0, cmap="viridis")
    ax.set_xticks(range(len(NAMES)), NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(NAMES)), NAMES, fontsize=8)
    ax.set_title(f"{key[1]}  [{key[0]}]", fontsize=11)
    for i in range(len(NAMES)):
        for j in range(len(NAMES)):
            v = A[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="w" if v < 0.6 else "k")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(OUT_DIR / "G8_similarity_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 9 — Stratified means

Pooled means over a population containing a degenerate space are the exact failure the
project already recorded. Within-family / cross-family / cross-modal are reported
separately, each with and without the flagged encoders.

In [ ]:
def strat(A):
    out = {}
    for label, pred in {
        "within-family":  lambda a, b: META[a]["family"] == META[b]["family"],
        "cross-family":   lambda a, b: META[a]["family"] != META[b]["family"] and META[a]["modality"] == META[b]["modality"],
        "cross-modal":    lambda a, b: META[a]["modality"] != META[b]["modality"],
    }.items():
        for tag, keep in {"all": lambda a, b: True,
                          "healthy-only": lambda a, b: a not in FLAGGED and b not in FLAGGED}.items():
            vals = [A.loc[a, b] for a, b in itertools.combinations(NAMES, 2)
                    if pred(a, b) and keep(a, b) and not np.isnan(A.loc[a, b])]
            out[f"{label} ({tag})"] = (round(float(np.mean(vals)), 4), len(vals)) if vals else (np.nan, 0)
    return out

summary = pd.DataFrame({f"{s} [{v}]": strat(A) for (v, s), A in M.items()
                        if s != "cka_biased"}).T
print(summary.to_string())
summary.to_csv(OUT_DIR / "G8_stratified_means.csv")

## 10 — Optional: does any of these predict hub transfer?

E.6 and E.12 established that raw shape agreement does **not** — ConvNeXt has the lowest ρ
of any image encoder (0.330–0.397) and the highest transfer (96.7%). This cell asks the
same question of mutual k-NN, debiased CKA and relative representations. Fill in
`TRANSFER` from the report's published column; a positive result here would be a new
finding and should be flagged as one, not folded in quietly.

In [ ]:
# published % of a natively-fitted head (report C.13 / E.2 / E.12) -- roster encoders only.
# SigLIP (94.2%) is NOT in this roster, so it is not listed here; use the held-in encoders.
TRANSFER = {
    "img_base": 95.9,
    "img_large": 92.9,
    "convnext": 96.7,
}
SOURCE = "img_small"      # the encoder the head was trained on

from scipy.stats import spearmanr, pearsonr

targets = [t for t in TRANSFER if t in NAMES and t != SOURCE]
print(f"source = {SOURCE}; targets = {targets}  (n = {len(targets)})")
if len(targets) < 4:
    print("n < 4 -- report the ordering, not a correlation coefficient")

rowsP = []
for (view, sname), A in M.items():
    sim = [A.loc[SOURCE, t] for t in targets]
    tr = [TRANSFER[t] for t in targets]
    if len(targets) >= 3 and not any(np.isnan(sim)):
        rowsP.append(dict(view=view, stat=sname,
                          spearman=round(float(spearmanr(sim, tr).statistic), 3),
                          pearson=round(float(pearsonr(sim, tr)[0]), 3),
                          sim=[round(s, 3) for s in sim]))
pred = pd.DataFrame(rowsP)
print(pred.to_string(index=False) if len(pred) else "not enough targets present")
print("\nWith four points every coefficient here is descriptive. The claim worth making is"
      "\nordinal: does the metric rank ConvNeXt above the DINOv2 targets, as transfer does?")

## 11 — Persist

In [ ]:
import json
with pd.ExcelWriter(OUT_DIR / "G8_matrices.xlsx") as xw:
    for (view, sname), A in M.items():
        A.to_excel(xw, sheet_name=f"{sname[:20]}_{view[:5]}")
    geo.to_excel(xw, sheet_name="geometry", index=False)
    gate.to_excel(xw, sheet_name="alignment_gate", index=False)
json.dump(dict(k=K_NN, anchors=N_ANCHORS, subsample=SUBSAMPLE, whiten_dim=WHITEN_DIM,
               n_rows=int(N), encoders=NAMES, flagged=sorted(FLAGGED),
               blocked=[f"{a}~{b}" for a, b in blocked],
               knn_chance=mutual_knn_chance(len(idx), K_NN)),
          open(OUT_DIR / "G8_config.json", "w"), indent=2)
print("written to", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print("   ", p.name)

## 12 — More views: clustered heatmap, metric agreement, distance-above-floor

Three additional figures. (a) The shape-agreement matrix with encoders reordered by hierarchical
clustering, so families fall into blocks. (b) A scatter of every encoder pair under two metrics,
to show where mutual-kNN and shape-ρ agree or diverge. (c) A heatmap of how many shuffle-SDs each
raw agreement sits above its own null — the honest "is this real" view.

In [ ]:
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

# (a) clustered heatmap of raw shape agreement
A = M[("raw", "shape_rho")].loc[NAMES, NAMES].values.astype(float)
D = 1.0 - np.clip(A, -1, 1); np.fill_diagonal(D, 0.0); D = (D + D.T) / 2
Z = linkage(squareform(D, checks=False), method="average")
order = dendrogram(Z, labels=NAMES, no_plot=True)["ivl"]
Ao = M[("raw", "shape_rho")].loc[order, order].values.astype(float)

fig = plt.figure(figsize=(13, 5.5))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.25], wspace=0.35)
axd = fig.add_subplot(gs[0, 0]); dendrogram(Z, labels=NAMES, ax=axd, color_threshold=0.0,
                                             leaf_rotation=90, above_threshold_color="#555")
axd.set_title("encoder clustering (1 - shape rho)"); axd.set_ylabel("distance")
axh = fig.add_subplot(gs[0, 1])
im = axh.imshow(Ao, vmin=np.nanmin(Ao), vmax=1.0, cmap="magma")
axh.set_xticks(range(len(order)), order, rotation=45, ha="right", fontsize=8)
axh.set_yticks(range(len(order)), order, fontsize=8)
for i in range(len(order)):
    for j in range(len(order)):
        axh.text(j, i, f"{Ao[i,j]:.2f}", ha="center", va="center", fontsize=7,
                 color="w" if Ao[i, j] < 0.6 else "k")
axh.set_title("shape agreement, clustered order"); fig.colorbar(im, ax=axh, fraction=0.046)
fig.savefig(OUT_DIR / "G8_clustered_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# (b) metric-agreement scatter: mutual-kNN vs shape-rho, one point per encoder pair
import itertools
knn = M[("raw", "mutual_knn")]; rho = M[("raw", "shape_rho")]
xs, ys, labs, cols = [], [], [], []
def _cls(a, b):
    if META[a]["family"] == META[b]["family"]: return "within-family", "#1f77b4"
    if META[a]["modality"] == META[b]["modality"]: return "cross-family", "#ff7f0e"
    return "cross-modal", "#2ca02c"
for a, b in itertools.combinations(NAMES, 2):
    xs.append(knn.loc[a, b]); ys.append(rho.loc[a, b])
    c, col = _cls(a, b); labs.append(c); cols.append(col)
fig, ax = plt.subplots(figsize=(7.5, 6))
for c in ["within-family", "cross-family", "cross-modal"]:
    m = [i for i, l in enumerate(labs) if l == c]
    ax.scatter([xs[i] for i in m], [ys[i] for i in m], s=55, alpha=0.8,
               c=[cols[i] for i in m], label=c, edgecolor="k", linewidth=0.4)
for a, b in itertools.combinations(NAMES, 2):
    if META[a]["modality"] != META[b]["modality"] or "gpt2" in (a, b):
        ax.annotate(f"{a}~{b}", (knn.loc[a, b], rho.loc[a, b]), fontsize=6, alpha=0.7)
ax.set_xlabel("mutual k-NN (raw)"); ax.set_ylabel("shape agreement rho (raw)")
ax.set_title("Do the two metrics agree? One point per encoder pair"); ax.legend()
fig.savefig(OUT_DIR / "G8_metric_scatter.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# (c) distance-above-floor heatmap: (raw agreement - shuffle mean) / shuffle SD, shape_rho.
# The null SD must be estimated from ENOUGH reps, each on a FRESH item sample, or a coincidentally
# tiny denominator inflates the ratio. n_rep=20 with per-rep resampling; the ratio is then a stable
# effect size, but still read the panel as CONFIRMATION (all pairs hugely > 0), not a fine ranking.
import itertools
ABOVE_REP = 20
Araw = M[("raw", "shape_rho")]
Zmat = pd.DataFrame(np.zeros((len(NAMES), len(NAMES))), index=NAMES, columns=NAMES, dtype=float)
for a, b in itertools.combinations(NAMES, 2):
    if (a, b) in blocked:
        Zmat.loc[a, b] = Zmat.loc[b, a] = np.nan; continue
    # each rep: a fresh item subsample AND a fresh row shuffle, so the null spread is real
    def _shuf(X, Y, _seed):
        rr = np.random.default_rng(_seed)
        idx = rr.choice(X.shape[0], 1500, replace=False)
        return shape_agreement(X, Y, sample=1500, rng=np.random.default_rng(int(idx[0])))
    vals = []
    rr = np.random.default_rng(hash((a, b)) & 0xffff)
    for _ in range(ABOVE_REP):
        perm = rr.permutation(E[b].shape[0])
        vals.append(_shuf(E[a], E[b][perm], rr.integers(1 << 30)))
    m, sd = float(np.mean(vals)), float(np.std(vals, ddof=1))
    z = (Araw.loc[a, b] - m) / (sd + 1e-9)
    Zmat.loc[a, b] = Zmat.loc[b, a] = z
for _d in NAMES: Zmat.loc[_d, _d] = np.nan

V = Zmat.values.astype(float)
# cap the colour scale at the 90th percentile so one huge value doesn't wash the panel out;
# annotate the true number regardless
vmax = float(np.nanpercentile(V, 90))
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(V, cmap="RdYlGn", vmin=0, vmax=vmax)
ax.set_xticks(range(len(NAMES)), NAMES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(NAMES)), NAMES, fontsize=8)
for i in range(len(NAMES)):
    for j in range(len(NAMES)):
        if not np.isnan(V[i, j]):
            ax.text(j, i, f"{V[i,j]:.0f}", ha="center", va="center", fontsize=7)
ax.set_title("shape agreement above its shuffle floor (SDs, n_rep=20)\nall pairs vastly > 0: this CONFIRMS realness, it is not a fine ranking")
fig.colorbar(im, ax=ax, fraction=0.046, label=f"SDs above null (scale capped at p90={vmax:.0f})")
fig.savefig(OUT_DIR / "G8_above_floor_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()
print("saved G8_clustered_heatmap.png, G8_metric_scatter.png, G8_above_floor_heatmap.png")
print("note: SD-above-null confirms every pair is real; it does not rank pairs — use the "
      "shape_rho matrix itself for that.")

---
## PASTE BACK FOR VERIFICATION — G8

After running, copy these **printed blocks** and **images** into the chat so I can confirm the fixes:

**Printed text (copy the whole block):**
1. The **geometry table** — the cell that prints `flagged degenerate: {...}` with the
   `mean_pair_cos / low_rank / anisotropic / degenerate` columns. *(I need to see that ONLY `gpt2`
   is flagged now — bge/bert/sbert must read `anisotropic=False`.)*
2. The **stratified means table** — the `within-family / cross-family / cross-modal` block with
   `(all)` and `(healthy-only)` columns. *(Cross-modal healthy-only should be ~0.288 again, over 7
   healthy encoders, not 3.)*

**Images (upload these files from `G8_outputs/`):**
- `G8_similarity_matrices.png`
- `G8_clustered_heatmap.png`
- `G8_above_floor_heatmap.png`

That's enough to verify G8. (The scatter and matrices are unchanged from before; the two tables
above are the only things the fix touched.)